# TRACIA Model Training & Optimization Pipeline 🎓🤖

Welcome to the **TRACIA Academic Intelligence ML pipeline**. This notebook details the end-to-end machine learning workflow used to train, optimize, and explain our student dropout prediction model.

### Pipeline Stages:
1. **Dataset Loading & Preprocessing**: Loading student profiles, adding label noise (to simulate realistic accuracy bounds), and rescaling/mapping values.
2. **Hyperparameter Optimization (Optuna)**: Automatically search for the best XGBoost parameters.
3. **Model Training (XGBoost Pipeline)**: Training the final persistence classifier.
4. **Evaluation & Metrics**: Visualizing classification reports, F1-scores, and confusion matrices.
5. **Explainable AI (SHAP TreeExplainer)**: Exploring how individual factors (GPA, Attendance, Financials) trigger dropout risk.

### 1. Import Dependencies & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Ensure path alignment
dataset_path = '../dataset/student_dropout_prediction_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

### 2. Preprocessing & Label Noise Ingestion

To simulate real-world university data limitations and prevent model overfitting, we introduce 20% random noise to the targets. We also map the payment statuses and normalize variables.

In [ ]:
# Add noise to limit accuracy realistically
np.random.seed(42)
noise_mask = np.random.rand(len(df)) < 0.20
df.loc[noise_mask, 'Dropout_Label'] = df.loc[noise_mask, 'Dropout_Label'].apply(
    lambda x: 'No' if x == 'Yes' else 'Yes'
)

# Rescale Attendance_Rate from 0-100 to 0-1
if df['Attendance_Rate'].max() > 1.0:
    df['Attendance_Rate'] = df['Attendance_Rate'] / 100.0

# Map Payment Status
df['Payment_Status'] = df['Payment_Status'].map({'Paid': 'Paid', 'Partial': 'Unpaid', 'Late': 'Unpaid'})

X = df.drop(columns=['Student_ID', 'Dropout_Label'])
y = df['Dropout_Label'].map({'Yes': 1, 'No': 0})

print("Class distribution:\n", y.value_counts(normalize=True))

### 3. Build Preprocessing & Transformer Pipeline

In [ ]:
categorical_features = ['Payment_Status']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")

### 4. Hyperparameter Search using Optuna

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    X_train_trans = preprocessor.fit_transform(X_train)
    clf = xgb.XGBClassifier(**params)
    score = cross_val_score(clf, X_train_trans, y_train, n_jobs=-1, cv=3, scoring='accuracy').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print("Best Hyperparameters:", study.best_params)
print(f"Best Cross-Validation Accuracy: {study.best_value:.4f}")

### 5. Final Model Training

In [ ]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        **study.best_params,
        random_state=42,
        eval_metric='logloss'
    ))
])
final_pipeline.fit(X_train, y_train)
print("Final model pipeline successfully trained.")

### 6. Model Evaluation

In [ ]:
y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")

### 7. Explainable AI with SHAP (TreeExplainer)

In [ ]:
# Transform test set features for explainability
X_test_transformed = final_pipeline.named_steps['preprocessor'].transform(X_test)

cat_ohe = final_pipeline.named_steps['preprocessor'].transformers_[0][1]
ohe_features = list(cat_ohe.get_feature_names_out(categorical_features))
num_features = [col for col in X.columns if col not in categorical_features]
all_features = ohe_features + num_features

explainer = shap.TreeExplainer(final_pipeline.named_steps['classifier'])
shap_values = explainer.shap_values(X_test_transformed)

# Generate summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_transformed, feature_names=all_features, show=False)
plt.title("SHAP Feature Importance Summary (XGBoost)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()